# Harmonischer Oszillator im Zustandsraum

Numerische Lösung mit dem Euler- und dem Verlet-Verfahren.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import Video, display

# Verzeichnis für manuell exportierte Abbildungen
notebook_name = "StateSpaceIllustration"
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / ".git").exists()),
    Path.cwd(),
)
output_dir = project_root / "data" / "01_mechanik" / notebook_name
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")

# Abbildungsstil für A4-Skripte: Einzelabbildungen bei etwa 0.5\textwidth.
latex_textwidth_in = 6.3
single_panel_width = 0.5 * latex_textwidth_in
single_panel_square = (single_panel_width, single_panel_width)
single_panel_wide = (single_panel_width, 0.68 * single_panel_width)
animation_display_width = 360  # Pixelbreite der eingebetteten Videos

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8.5,
    "lines.linewidth": 1.2,
    "axes.linewidth": 0.8,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.03,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

## Bewegungsgleichung

Für einen harmonischen Oszillator gilt

$$\ddot{x} = -\omega^2 x. $$

Mit $v=\dot{x}$ ergibt sich das System erster Ordnung

$$\dot{x}=v, \qquad \dot{v}=-\omega^2x. $$

Der **Zustandsraum** wird hier von den Variablen $x$ und $\dot{x}$ aufgespannt. Die exakte Lösung beschreibt darin eine geschlossene Kurve. Für $\omega=1$ ist diese Kurve ein Kreis.

In [ ]:
def acceleration(x, omega):
    return -(omega**2) * x


def euler_step(x, v, h, omega):
    x_new = x + h * v
    v_new = v + h * acceleration(x, omega)
    return x_new, v_new


def verlet_step(x, v, h, omega):
    v_half = v + 0.5 * h * acceleration(x, omega)
    x_new = x + h * v_half
    v_new = v_half + 0.5 * h * acceleration(x_new, omega)
    return x_new, v_new


def solve(step, x0, v0, h, n_steps, omega=1.0):
    t = h * np.arange(n_steps + 1)
    x = np.empty(n_steps + 1)
    v = np.empty(n_steps + 1)
    x[0], v[0] = x0, v0

    for n in range(n_steps):
        x[n + 1], v[n + 1] = step(x[n], v[n], h, omega)

    return t, x, v

In [ ]:
# Anfangswerte und Zeitschritt
omega = 1.0
x0 = 1.0
v0 = 0.0
h = 0.1
n_steps = 200

t, x_euler, v_euler = solve(euler_step, x0, v0, h, n_steps, omega)
_, x_verlet, v_verlet = solve(verlet_step, x0, v0, h, n_steps, omega)

# Exakte Lösung zum Vergleich
x_exact = x0 * np.cos(omega * t) + (v0 / omega) * np.sin(omega * t)
v_exact = -omega * x0 * np.sin(omega * t) + v0 * np.cos(omega * t)

In [ ]:
fig_time, ax = plt.subplots(figsize=single_panel_wide)
ax.plot(t, x_exact, color="black", label="exakt")
ax.plot(t, x_euler, "--", label="Euler")
ax.plot(t, x_verlet, label="Verlet")
ax.set(xlabel="Zeit $t$", ylabel="Auslenkung $x(t)$")
ax.grid(alpha=0.3)
ax.legend()
fig_time.tight_layout()
plt.show()

# Manueller Export, falls gewünscht:
# fig_time.savefig(output_dir / "zeitentwicklung.pdf", bbox_inches="tight")

## Darstellung im Zustandsraum

Die Erhaltung der Energie

$$E=\frac{1}{2}\dot{x}^2+\frac{1}{2}\omega^2x^2$$

führt zu geschlossenen Bahnen im Zustandsraum. Das explizite Euler-Verfahren vergrößert dagegen die Energie systematisch, während das Verlet-Verfahren die geschlossene Bahn gut erhält.

In [ ]:
# Exakte Energiekonturen und eine mit Verlet berechnete Bahn
x_grid, v_grid = np.meshgrid(
    np.linspace(-2.0, 2.0, 300),
    np.linspace(-2.0, 2.0, 300),
)
energy_grid = 0.5 * v_grid**2 + 0.5 * omega**2 * x_grid**2
energy_levels = np.linspace(0.1, 2.0, 10)

fig_energy, ax = plt.subplots(figsize=single_panel_square)
contours = ax.contour(
    x_grid, v_grid, energy_grid, levels=energy_levels,
    colors="0.75", linewidths=1.0,
)
ax.clabel(contours, inline=True, fontsize=8, fmt="E = %.1f")
ax.plot(x_verlet[:40], v_verlet[:40], color="C0", linewidth=2, label="Trajektorie")
ax.plot(x0, v0, "o", color="C0", label="Anfangszustand")
ax.set(
    xlabel="Auslenkung $x$",
    ylabel="Geschwindigkeit $\dot{x}$",
    #title="Energiekonturen im Zustandsraum",
)
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.2)
ax.legend()
fig_energy.tight_layout()
plt.show()

fig_energy.savefig(output_dir / "energiekonturen_verlet.pdf", bbox_inches="tight")

In [ ]:
fig_state, ax = plt.subplots(figsize=single_panel_square)
ax.plot(x_exact, v_exact, color="black", label="exakt")
ax.plot(x_euler, v_euler, "--", label="Euler")
ax.plot(x_verlet, v_verlet, label="Verlet")
ax.plot(x0, v0, "ko", label="Anfangszustand")
ax.set(
    xlabel="Auslenkung $x$",
    ylabel="Geschwindigkeit $\dot{x}$",
    #title="Harmonischer Oszillator\nim Zustandsraum",
)
ax.axhline(0, color="0.8", linewidth=0.8)
ax.axvline(0, color="0.8", linewidth=0.8)
ax.grid(alpha=0.3)
ax.legend()
fig_state.tight_layout()
plt.show()

# Manueller Export, falls gewünscht:
fig_state.savefig(output_dir / "zustandsraum.pdf", bbox_inches="tight")

## Fluss im Zustandsraum

Die Punkte umschließen zu Beginn eine kleine Fläche im Zustandsraum. Jeder Punkt wird mit dem Verlet-Verfahren weiterentwickelt. Für den ungedämpften harmonischen Oszillator bleibt die eingeschlossene Fläche erhalten; sie wird lediglich im Zustandsraum bewegt und leicht verformt.

In [ ]:
# Geschlossene Kurve aus Anfangszuständen
n_boundary = 20
angles = np.linspace(0, 2 * np.pi, n_boundary, endpoint=False)
x_center, v_center = 0.9, 0.0
radius = 0.22
x_boundary = x_center + radius * np.cos(angles)
v_boundary = v_center + radius * np.sin(angles)

h_flow = 0.05
n_flow_steps = 160
x_flow = np.empty((n_flow_steps + 1, n_boundary))
v_flow = np.empty_like(x_flow)
x_flow[0], v_flow[0] = x_boundary, v_boundary

for n in range(n_flow_steps):
    x_flow[n + 1], v_flow[n + 1] = verlet_step(
        x_flow[n], v_flow[n], h_flow, omega
    )

In [ ]:
# Die umschlossene Fläche zu ausgewählten diskreten Zeiten
snapshot_indices = [0, 10, 20, 30, 40]
fig_flow, ax = plt.subplots(figsize=single_panel_square)
ax.contour(x_grid, v_grid, energy_grid, levels=energy_levels, colors="0.88")

for index in snapshot_indices:
    closed_x = np.r_[x_flow[index], x_flow[index, 0]]
    closed_v = np.r_[v_flow[index], v_flow[index, 0]]
    ax.plot(
        closed_x, closed_v, ".-", markersize=5, linewidth=1,
        label=fr"$t={index * h_flow:.1f}$",
    )

ax.set(
    xlabel="Auslenkung $x$",
    ylabel="Geschwindigkeit $\dot{x}$",
    #title="Fluss einer Fläche im\nZustandsraum (Verlet)",
)
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-1.4, 1.4)
ax.set_ylim(-1.4, 1.4)
ax.grid(alpha=0.2)
ax.legend(ncols=2)
fig_flow.tight_layout()
plt.show()

fig_flow.savefig(output_dir / "zustandsraumfluss_verlet.pdf", bbox_inches="tight")

In [ ]:
# Animation des Zustandsraumflusses
fig, ax = plt.subplots(figsize=single_panel_square)
ax.contour(x_grid, v_grid, energy_grid, levels=energy_levels, colors="0.88")
flow_line, = ax.plot([], [], ".-", color="C0", markersize=3, linewidth=1)
time_text = ax.text(0.03, 0.95, "", transform=ax.transAxes, va="top")
ax.set(
    xlabel="Auslenkung $x$",
    ylabel="Geschwindigkeit $\dot{x}$",
    # title="Fluss im Zustandsraum\n(Verlet)",
    xlim=(-1.4, 1.4), ylim=(-1.4, 1.4),
)
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.2)

def update_flow(frame):
    flow_line.set_data(
        np.r_[x_flow[frame], x_flow[frame, 0]],
        np.r_[v_flow[frame], v_flow[frame, 0]],
    )
    time_text.set_text(f"t = {frame * h_flow:.2f}")
    return flow_line, time_text

flow_animation = FuncAnimation(
    fig, update_flow, frames=n_flow_steps + 1, interval=40, blit=True
)
flow_video = output_dir / "zustandsraumfluss_verlet.mp4"
flow_animation.save(str(flow_video), writer="ffmpeg", fps=25)
plt.close(fig)
display(Video(
    str(flow_video), embed=True,
    html_attributes=f'controls loop muted width="{animation_display_width}"',
))

## Dissipativer harmonischer Oszillator

Für den gedämpften Oszillator gilt

$$\ddot{x} + \gamma\dot{x} + \omega^2x = 0. $$

Die Zustände werden hier mit einem Runge-Kutta-Verfahren vierter Ordnung (RK4) entwickelt. Wegen der Dissipation schrumpft die von den Punkten eingeschlossene Fläche: Der Zustandsraumfluss ist nicht flächenerhaltend.

In [ ]:
def dissipative_rhs(state, omega, gamma):
    x, v = state
    return np.array([v, -(omega**2) * x - gamma * v])


def rk4_step(state, h, omega, gamma):
    k1 = dissipative_rhs(state, omega, gamma)
    k2 = dissipative_rhs(state + 0.5 * h * k1, omega, gamma)
    k3 = dissipative_rhs(state + 0.5 * h * k2, omega, gamma)
    k4 = dissipative_rhs(state + h * k3, omega, gamma)
    return state + (h / 6) * (k1 + 2 * k2 + 2 * k3 + k4)


gamma = 0.25
h_flow = 0.05
n_flow_steps = 320
dissipative_flow = np.empty((n_flow_steps + 1, 2, n_boundary))
dissipative_flow[0] = np.array([x_boundary, v_boundary])

for n in range(n_flow_steps):
    dissipative_flow[n + 1] = rk4_step(
        dissipative_flow[n], h_flow, omega, gamma
    )

x_diss = dissipative_flow[:, 0]
v_diss = dissipative_flow[:, 1]

In [ ]:
# Dissipativer Zustandsraumfluss zu ausgewählten diskreten Zeiten
dissipative_snapshot_indices = np.linspace(0, n_flow_steps, 17, dtype=int)
fig_diss, ax = plt.subplots(figsize=single_panel_square)
time_norm = plt.Normalize(0, n_flow_steps * h_flow)
time_cmap = plt.colormaps["viridis"]

for index in dissipative_snapshot_indices:
    closed_x = np.r_[x_diss[index], x_diss[index, 0]]
    closed_v = np.r_[v_diss[index], v_diss[index, 0]]
    ax.plot(
        closed_x, closed_v, "-", linewidth=1,
        color=time_cmap(time_norm(index * h_flow)),
    )

ax.set(
    xlabel="Auslenkung $x$",
    ylabel="Geschwindigkeit $\dot{x}$",
    # title=fr"Dissipativer Zustandsraumfluss ($\gamma={gamma}$, RK4)",
    xlim=(-1.4, 1.4), ylim=(-1.4, 1.4),
)
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.2)
time_map = plt.cm.ScalarMappable(norm=time_norm, cmap=time_cmap)
colorbar = fig_diss.colorbar(time_map, ax=ax, pad=0.03)
colorbar.set_label("Zeit $t$")
fig_diss.tight_layout()
plt.show()

fig_diss.savefig(output_dir / "zustandsraumfluss_dissipativ.pdf", bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=single_panel_square)
ax.contour(x_grid, v_grid, energy_grid, levels=energy_levels, colors="0.88")
flow_line, = ax.plot([], [], ".-", color="C0", markersize=3, linewidth=1)
time_text = ax.text(0.03, 0.95, "", transform=ax.transAxes, va="top")

diss_line, = ax.plot([], [], ".-", color="C3", markersize=3, linewidth=1)
time_text = ax.text(0.03, 0.95, "", transform=ax.transAxes, va="top")
ax.set(
    xlabel="Auslenkung $x$",
    ylabel="Geschwindigkeit $\dot{x}$",
    # title=fr"Dissipativer Zustandsraumfluss\n($\gamma={gamma}$, RK4)",
    xlim=(-1.4, 1.4), ylim=(-1.4, 1.4),
)
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.2)

def update_dissipative_flow(frame):
    diss_line.set_data(
        np.r_[x_diss[frame], x_diss[frame, 0]],
        np.r_[v_diss[frame], v_diss[frame, 0]],
    )
    time_text.set_text(f"t = {frame * h_flow:.2f}")
    return diss_line, time_text

dissipative_animation = FuncAnimation(
    fig, update_dissipative_flow, frames=n_flow_steps + 1, interval=40, blit=True
)
dissipative_video = output_dir / "zustandsraumfluss_dissipativ.mp4"
dissipative_animation.save(str(dissipative_video), writer="ffmpeg", fps=25)
plt.close(fig)
display(Video(
    str(dissipative_video), embed=True,
    html_attributes=f'controls loop muted width="{animation_display_width}"',
))